# Geração de Tabela LaTeX a partir de arquivos .txt

Este notebook lê todos os arquivos `.txt` da pasta `input`, processa os dados e gera uma tabela em formato LaTeX para ser utilizada no relatório.

## 1. Importar bibliotecas necessárias
Vamos importar as bibliotecas pandas e os para manipulação de arquivos e dados.

In [1]:
import pandas as pd
import os
from pathlib import Path

## 2. Estrutura de pastas

In [2]:
from pathlib import Path
import shutil

# Caminhos absolutos (ajuste conforme necessário)
src_base = Path.cwd().parent / 'src' / 'dados'
dst_base = Path.cwd().parent / 'relatorioCefet' / 'tabelas'

# Remove a pasta de destino inteira se existir (limpeza total)
if dst_base.exists() and dst_base.is_dir():
    shutil.rmtree(dst_base)
    print(f'Pasta de destino removida: {dst_base}')

# Cria input e output novamente
(dst_base / 'input').mkdir(parents=True, exist_ok=True)
(dst_base / 'output').mkdir(parents=True, exist_ok=True)

# Cria as subpastas de output conforme existem em src/dados/output
output_src = src_base / 'output'
if output_src.exists():
    for subpasta in output_src.iterdir():
        if subpasta.is_dir():
            (dst_base / 'output' / subpasta.name).mkdir(parents=True, exist_ok=True)
print('Estrutura de pastas criada em:', dst_base)


Pasta de destino removida: d:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\PraticaQuedaLIvre\relatorioCefet\tabelas
Estrutura de pastas criada em: d:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\PraticaQuedaLIvre\relatorioCefet\tabelas


## 2. Listar arquivos .csv na pasta input
Vamos listar todos os arquivos `.csv` presentes na pasta `input`.

In [3]:
from pathlib import Path

# Define os caminhos das pastas
input_dir = Path('../src/dados/input').resolve()
output_dir = Path('../src/dados/output').resolve()

# Lista todos os arquivos em input (não recursivo)
arquivos_input = [arq for arq in input_dir.glob('*') if arq.is_file()]

# Lista todos os arquivos em output (recursivo, incluindo subpastas)
arquivos_output = [arq for arq in output_dir.rglob('*') if arq.is_file()]

print(f'Arquivos em input ({len(arquivos_input)}):')
for arq in arquivos_input:
    print(arq)

print(f'\nArquivos em output ({len(arquivos_output)}):')
for arq in arquivos_output:
    print(arq)

Arquivos em input (13):
D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\PraticaQuedaLIvre\src\dados\input\entrada_acuracia.csv
D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\PraticaQuedaLIvre\src\dados\input\entrada_queda_livre.csv
D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\PraticaQuedaLIvre\src\dados\input\entrada_queda_livre_linear_completo.csv
D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\PraticaQuedaLIvre\src\dados\input\entrada_queda_livre_linear_simplificada.csv
D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\PraticaQuedaLIvre\src\dados\input\entrada_queda_livre_quadratica_completo.csv
D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\PraticaQuedaLIvre\src\dados\input\entrada_queda_livre_quadratica_simplificada.csv
D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\PraticaQuedaLIvre\src\dados\input\entrada_queda_livre_resistencia_ar.csv
D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\PraticaQuedaLIvre\src\dados\input\parametros_a

## 3. Ler e processar arquivos .csv
Vamos ler o conteúdo de cada arquivo `.csv` e armazenar os dados em uma lista.

In [4]:
import pandas as pd
from pathlib import Path

# ============================================================
# Funções auxiliares
# ============================================================

def escapar_latex(texto):
    """Escapa caracteres especiais para LaTeX."""
    return (str(texto)
            .replace('_', '\\_')
            .replace('%', '\\%')
            .replace('&', '\\&')
            .replace('#', '\\#')
            .replace('{', '\\{')
            .replace('}', '\\}')
            .replace('^', '\\^{}')
            )

def gerar_caption(nome_base):
    """Gera caption padronizada a partir do nome do arquivo."""
    nome = nome_base.replace('_', ' ').replace('-', ' ')
    nome = nome.capitalize()
    return f"Tabela – Dados referentes a {nome}."

def gerar_label(nome_base):
    """Gera label padronizado."""
    nome = nome_base.replace('.', '_').replace('-', '_')
    return f"tab:{nome}"

def salvar_tabela_latex(arquivo_csv, pasta_saida):
    """Converte CSV em tabela LaTeX com caption e label."""
    try:
        df = pd.read_csv(arquivo_csv)

        # Escapa caracteres especiais
        df_latex = df.copy()
        df_latex.columns = [escapar_latex(col) for col in df_latex.columns]
        for col in df_latex.columns:
            df_latex[col] = df_latex[col].astype(str).apply(escapar_latex)

        # Nome base
        nome_base = arquivo_csv.stem
        caption = gerar_caption(nome_base)
        label = gerar_label(nome_base)

        # Tabela LaTeX
        tabela = df_latex.to_latex(index=False, escape=False)

        # Ambiente completo
        conteudo = (
            "\\begin{table}[H]\n"
            "\\centering\n"
            f"\\caption{{{caption}}}\n"
            f"\\label{{{label}}}\n"
            f"{tabela}\n"
            "\\end{table}\n"
        )

        # Caminho final
        nome_saida = arquivo_csv.with_suffix('.tex').name
        caminho_saida = pasta_saida / nome_saida
        pasta_saida.mkdir(parents=True, exist_ok=True)

        with open(caminho_saida, 'w', encoding='utf-8') as f:
            f.write(conteudo)

        print(f"Tabela LaTeX salva em: {caminho_saida}")

    except Exception as e:
        print(f"Erro ao processar {arquivo_csv}: {e}")


# ============================================================
# Processamento dos arquivos de INPUT
# ============================================================

pasta_saida_input = Path('../relatorioCefet/tabelas/input').resolve()

if not arquivos_input:
    print("Nenhum arquivo encontrado em input.")
else:
    for arquivo in arquivos_input:
        salvar_tabela_latex(arquivo, pasta_saida_input)


# ============================================================
# Processamento dos arquivos de OUTPUT (mantendo subpastas)
# ============================================================

pasta_saida_output = Path('../relatorioCefet/tabelas/output').resolve()
output_dir = Path('../src/dados/output').resolve()

if not arquivos_output:
    print("Nenhum arquivo encontrado em output.")
else:
    for arquivo in arquivos_output:
        subpath = arquivo.relative_to(output_dir).parent
        destino = pasta_saida_output / subpath
        destino.mkdir(parents=True, exist_ok=True)
        salvar_tabela_latex(arquivo, destino)

print("Processo finalizado.")

Tabela LaTeX salva em: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\PraticaQuedaLIvre\relatorioCefet\tabelas\input\entrada_acuracia.tex
Tabela LaTeX salva em: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\PraticaQuedaLIvre\relatorioCefet\tabelas\input\entrada_queda_livre.tex
Tabela LaTeX salva em: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\PraticaQuedaLIvre\relatorioCefet\tabelas\input\entrada_queda_livre_linear_completo.tex
Tabela LaTeX salva em: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\PraticaQuedaLIvre\relatorioCefet\tabelas\input\entrada_queda_livre_linear_simplificada.tex
Tabela LaTeX salva em: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\PraticaQuedaLIvre\relatorioCefet\tabelas\input\entrada_queda_livre_quadratica_completo.tex
Tabela LaTeX salva em: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\PraticaQuedaLIvre\relatorioCefet\tabelas\input\entrada_queda_livre_quadratica_simplificada.tex
Tabela LaTeX salva em: D:\GitHub\Do

## 8. Copiar toda a pasta de gráficos para figuras

Esta célula copia recursivamente todo o conteúdo da pasta `graficos` (incluindo subpastas e arquivos) para a pasta `relatorioCefet/figuras`. Útil para garantir que todos os gráficos estejam disponíveis no relatório.

In [5]:
import shutil
from pathlib import Path

# Caminhos de origem e destino
graficos_dir = Path('../src/graficos').resolve()
figuras_dir = Path('../relatorioCefet/figuras').resolve()
destino_graficos = figuras_dir / 'graficos'

def copiar_pasta(origem, destino):
    # Remove a pasta de destino se já existir
    if destino.exists() and destino.is_dir():
        shutil.rmtree(destino)
        print(f'Pasta de destino removida: {destino}')
    # Cria a pasta de destino novamente
    destino.mkdir(parents=True, exist_ok=True)
    if not origem.exists():
        print(f'Pasta de origem não existe: {origem}')
        return
    for item in origem.rglob('*'):
        if item.is_file():
            destino_arquivo = destino / item.relative_to(origem)
            destino_arquivo.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(item, destino_arquivo)
            print(f'Arquivo copiado: {item} -> {destino_arquivo}')

copiar_pasta(graficos_dir, destino_graficos)


Pasta de destino removida: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\PraticaQuedaLIvre\relatorioCefet\figuras\graficos
Arquivo copiado: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\PraticaQuedaLIvre\src\graficos\acuracia\grafico_acuracia_com_resistencia_linear_c_0_0028787500000000002.png -> D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\PraticaQuedaLIvre\relatorioCefet\figuras\graficos\acuracia\grafico_acuracia_com_resistencia_linear_c_0_0028787500000000002.png
Arquivo copiado: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\PraticaQuedaLIvre\src\graficos\acuracia\grafico_acuracia_com_resistencia_linear_c_20_0.png -> D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\PraticaQuedaLIvre\relatorioCefet\figuras\graficos\acuracia\grafico_acuracia_com_resistencia_linear_c_20_0.png
Arquivo copiado: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\PraticaQuedaLIvre\src\graficos\acuracia\grafico_acuracia_com_resistencia_linear_c_2_0.png -> D:\GitHub\Dou

In [6]:
import shutil
from pathlib import Path

# Caminhos de origem e destino
graficos_dir = Path('../src/graficos').resolve()
figuras_dir = Path('../relatorioCefet/figuras').resolve()
destino_graficos = figuras_dir / 'graficos'

def copiar_pasta(origem, destino):
    if not origem.exists():
        print(f'Pasta de origem não existe: {origem}')
        return
    for item in origem.rglob('*'):
        if item.is_file():
            destino_arquivo = destino / item.relative_to(origem)
            destino_arquivo.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(item, destino_arquivo)
            print(f'Arquivo copiado: {item} -> {destino_arquivo}')

copiar_pasta(graficos_dir, destino_graficos)



Arquivo copiado: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\PraticaQuedaLIvre\src\graficos\acuracia\grafico_acuracia_com_resistencia_linear_c_0_0028787500000000002.png -> D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\PraticaQuedaLIvre\relatorioCefet\figuras\graficos\acuracia\grafico_acuracia_com_resistencia_linear_c_0_0028787500000000002.png
Arquivo copiado: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\PraticaQuedaLIvre\src\graficos\acuracia\grafico_acuracia_com_resistencia_linear_c_20_0.png -> D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\PraticaQuedaLIvre\relatorioCefet\figuras\graficos\acuracia\grafico_acuracia_com_resistencia_linear_c_20_0.png
Arquivo copiado: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\PraticaQuedaLIvre\src\graficos\acuracia\grafico_acuracia_com_resistencia_linear_c_2_0.png -> D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\PraticaQuedaLIvre\relatorioCefet\figuras\graficos\acuracia\grafico_acuracia_com_resistenci